# Teste Técnico – Pipeline Bronze → Silver → Gold (Databricks Free)
**Autor:** Victor Amichi  
**Objetivo:** Ingerir arquivos (CSV/Parquet) na Bronze, padronizar na Silver e responder perguntas de negócio na Gold.

**Camadas:**
- **Bronze:** espelha a fonte + `_ingest_ts`
- **Silver:** limpeza + tipagem básica do que importa
- **Gold:** métricas e queries de negócio (abandono de carrinhos)


## 0) Parâmetros (Catalog / Schema / Volumes)
Aqui centralizamos os caminhos para não ficar hard-coded no notebook.


In [0]:
# ===== PARAMETROS UC =====
CATALOG = "workspace"
SCHEMA  = "cantustore"

RAW_VOL    = "raw_prova_dados"
BRONZE_VOL = "bronze_prova_dados"
SILVER_VOL = "silver_prova_dados"

# ===== PATHS (DBFS/Volumes) =====
RAW    = f"dbfs:/Volumes/{CATALOG}/{SCHEMA}/{RAW_VOL}"
BRONZE = f"dbfs:/Volumes/{CATALOG}/{SCHEMA}/{BRONZE_VOL}"
SILVER = f"dbfs:/Volumes/{CATALOG}/{SCHEMA}/{SILVER_VOL}"

print("RAW   =", RAW)
print("BRONZE=", BRONZE)
print("SILVER=", SILVER)


## 1) Preparar volumes (Databricks Free / Unity Catalog)
Criamos os volumes de saída (Bronze e Silver).  
> Obs: você já tem o volume RAW.


In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{BRONZE_VOL}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{SILVER_VOL}")

dbutils.fs.mkdirs(BRONZE)
dbutils.fs.mkdirs(SILVER)

display(dbutils.fs.ls(RAW))


# BRONZE
## 2) Funções de ingestão (CSV / Parquet)
Bronze espelha a fonte e adiciona `_ingest_ts`.


In [0]:
from pyspark.sql import functions as F

def csv_to_bronze(table_name):
    src = f"{RAW}/{table_name}/*.csv"
    tgt = f"{BRONZE}/{table_name}"

    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .option("sep", "|")
          .option("quote", '"')
          .csv(src)
          .withColumn("_ingest_ts", F.current_timestamp()))

    df.write.format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .save(tgt)

    print("OK bronze csv:", table_name)

def parquet_to_bronze(table_name):
    src = f"{RAW}/{table_name}/*.parquet"
    tgt = f"{BRONZE}/{table_name}"

    df = spark.read.parquet(src).withColumn("_ingest_ts", F.current_timestamp())

    df.write.format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .save(tgt)

    print("OK bronze parquet:", table_name)


## 3) Descobrir tabelas no RAW e carregar na Bronze
O RAW tem pastas (1 pasta por tabela) e dentro pode ter CSV ou Parquet.


In [0]:
folders = [f.name.rstrip("/") for f in dbutils.fs.ls(RAW) if f.isDir()]
print("Tabelas encontradas:", folders)

for t in folders:
    files = dbutils.fs.ls(f"{RAW}/{t}")
    has_parquet = any(f.name.endswith(".parquet") for f in files)
    has_csv = any(f.name.endswith(".csv") for f in files)

    if has_parquet:
        parquet_to_bronze(t)
    elif has_csv:
        csv_to_bronze(t)
    else:
        print("SKIP (nenhum csv/parquet):", t)

display(dbutils.fs.ls(BRONZE))


# SILVER
## 4) Regras básicas de limpeza/tipagem
**Objetivo:** fazer o mínimo “padrão dia a dia”:
- datas → timestamp (com tolerância a lixo)
- colunas de chave → `long`
- valores numéricos importantes → arredondar (se quiser)


In [0]:
from pyspark.sql import functions as F

# Chaves (do seu diagrama / joins)
KEYS = {
  "tb_addresses":    ["PK", "p_region"],
  "tb_cartentries":  ["PK", "p_order"],
  "tb_carts":        ["PK", "p_user", "p_paymentinfo", "p_paymentmode", "p_deliveryaddress", "p_site"],
  "tb_cmssitelp":    ["ITEMPK"],
  "tb_paymentinfos": ["PK"],
  "tb_paymentmodes": ["PK"],
  "tb_regions":      ["PK"],
  "tb_users":        ["PK"]
}

# Datas que aparecem (em algumas tabelas vem string, em outras timestamp)
DATE_COLS = ["createdTS", "modifiedTS", "p_lastlogin", "p_registerdatetime"]

# Exemplo de colunas numéricas que você pode querer padronizar (opcional)
MONEY_2 = ["p_totalprice", "p_baseprice", "p_deliverycost", "p_paymentcost",
           "p_totaldiscounts", "p_totaltax", "p_subtotal", "p_subtotalservice", "p_subtotalwithoutdiscounts"]

QTY_3 = ["p_quantity"]


## 5) Função Bronze → Silver (simples e tolerante)
**Importante:** quando tiver lixo em coluna de data (ex.: `'1.00000000'`), não quebra: vira `NULL`.


In [0]:
import re
from pyspark.sql import functions as F

def safe_to_timestamp(col):
    """
    Converte string para timestamp apenas se parecer data.
    Se não parecer, retorna NULL (evita erro de cast).
    """
    # padrões simples (você pode ajustar)
    # 2021-10-17 23:44:00.250
    # 2021-10-17 23:44:00
    # 2021-10-17
    return F.when(
        F.col(col).cast("string").rlike(r"^\d{4}-\d{2}-\d{2}([ T]\d{2}:\d{2}:\d{2}(\.\d{1,3})?)?$"),
        F.to_timestamp(F.col(col).cast("string"))
    ).otherwise(F.lit(None).cast("timestamp"))

def bronze_to_silver(table_name):
    src = f"{BRONZE}/{table_name}"
    tgt = f"{SILVER}/{table_name}"

    df = spark.read.format("delta").load(src)

    # 1) Tipar chaves para long (se existirem)
    for k in KEYS.get(table_name, []):
        if k in df.columns:
            df = df.withColumn(k, F.col(k).cast("long"))

    # 2) Tipar datas (tolerante a dados ruins)
    for c in DATE_COLS:
        if c in df.columns:
            # se já for timestamp, mantém; se for string, tenta converter
            if dict(df.dtypes).get(c) != "timestamp":
                df = df.withColumn(c, safe_to_timestamp(c))

    # 3) Padronizar numéricos (opcional)
    for c in MONEY_2:
        if c in df.columns:
            df = df.withColumn(c, F.round(F.col(c).cast("double"), 2))
    for c in QTY_3:
        if c in df.columns:
            df = df.withColumn(c, F.round(F.col(c).cast("double"), 3))

    # 4) Escrever Silver
    df.write.format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .save(tgt)

    print("OK silver:", table_name)


## 6) Executar Bronze → Silver para todas as tabelas


In [0]:
tables = [f.name.rstrip("/") for f in dbutils.fs.ls(BRONZE)]
print("Tabelas Bronze:", tables)

for t in tables:
    bronze_to_silver(t)

display(dbutils.fs.ls(SILVER))


# GOLD (Análises)
## 7) Temp views “base” (reuso)
Vamos criar views que você usa em várias perguntas:
- `vw_base_carts`: carrinhos classificados (abandoned/completed)
- `vw_cart_product`: relação carrinho x produto (sem duplicar)
- `vw_abandoned_items`: itens apenas de carrinhos abandonados
- `vw_carts_with_state`: carrinhos + estado (endereço)


In [0]:
%sql
-- 7.1) Base carts (regra central: abandoned vs completed)
CREATE OR REPLACE TEMP VIEW vw_base_carts AS
SELECT
  c.*,
  CASE WHEN pi.PK IS NULL THEN 'abandoned' ELSE 'completed' END AS cart_status
FROM delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_carts` c
LEFT JOIN delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_paymentinfos` pi
  ON c.p_paymentinfo = pi.PK;


In [0]:
%sql
-- 7.2) Relação carrinho x produto (evita duplicar produto no mesmo carrinho)
CREATE OR REPLACE TEMP VIEW vw_cart_product AS
SELECT DISTINCT
  p_order   AS cart_id,
  p_product AS product_code
FROM delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_cartentries`;


In [0]:
%sql
-- 7.3) Itens apenas de carrinhos abandonados (reuso em várias análises)
CREATE OR REPLACE TEMP VIEW vw_abandoned_items AS
SELECT
  cp.cart_id,
  cp.product_code
FROM vw_cart_product cp
JOIN vw_base_carts bc
  ON cp.cart_id = bc.PK
WHERE bc.cart_status = 'abandoned';


In [0]:
%sql
-- 7.4) Carrinhos + estado (p_region) via endereço de entrega
CREATE OR REPLACE TEMP VIEW vw_carts_with_state AS
SELECT
  bc.PK AS cart_id,
  bc.cart_status,
  a.p_region AS state_code,
  a.p_town   AS city
FROM vw_base_carts bc
JOIN delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_addresses` a
  ON bc.p_deliveryaddress = a.PK;


## 8) 2.1 Produtos com mais carrinhos abandonados


In [0]:
%sql
SELECT
  product_code AS codigo_do_produto,
  COUNT(DISTINCT cart_id) AS carrinhos_abandonados
FROM vw_abandoned_items
GROUP BY product_code
ORDER BY carrinhos_abandonados DESC;


## 9) 2.2 Duplas de produtos mais comuns em carrinhos abandonados


In [0]:
%sql
WITH pairs AS (
  SELECT
    a.cart_id,
    LEAST(a.product_code, b.product_code)    AS product_a,
    GREATEST(a.product_code, b.product_code) AS product_b
  FROM vw_abandoned_items a
  JOIN vw_abandoned_items b
    ON a.cart_id = b.cart_id
   AND a.product_code < b.product_code
)
SELECT
  product_a,
  product_b,
  COUNT(DISTINCT cart_id) AS abandoned_carts
FROM pairs
GROUP BY product_a, product_b
ORDER BY abandoned_carts DESC;


## 10) 2.3 Produtos com aumento de abandono (mês a mês)
Criamos uma temp view com a taxa de abandono por produto/mês e sua variação.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_product_abandon_mom AS
WITH pm AS (
  SELECT
    cp.product_code,
    date_trunc('month', bc.createdTS) AS month,
    COUNT(DISTINCT bc.PK) AS total_carts,
    COUNT(DISTINCT CASE WHEN bc.cart_status = 'abandoned' THEN bc.PK END) AS abandoned_carts
  FROM vw_cart_product cp
  JOIN vw_base_carts bc
    ON cp.cart_id = bc.PK
  WHERE bc.createdTS IS NOT NULL
  GROUP BY cp.product_code, date_trunc('month', bc.createdTS)
),
rates AS (
  SELECT
    product_code,
    month,
    total_carts,
    abandoned_carts,
    abandoned_carts / NULLIF(total_carts, 0) AS abandon_rate
  FROM pm
)
SELECT
  product_code,
  month,
  total_carts,
  abandoned_carts,
  abandon_rate,
  LAG(abandon_rate) OVER (PARTITION BY product_code ORDER BY month) AS prev_abandon_rate,
  (abandon_rate - LAG(abandon_rate) OVER (PARTITION BY product_code ORDER BY month)) AS abandon_rate_diff,
  CASE
    WHEN LAG(abandon_rate) OVER (PARTITION BY product_code ORDER BY month) IS NULL THEN 'no_prev_month'
    WHEN (abandon_rate - LAG(abandon_rate) OVER (PARTITION BY product_code ORDER BY month)) > 0 THEN 'increase'
    WHEN (abandon_rate - LAG(abandon_rate) OVER (PARTITION BY product_code ORDER BY month)) < 0 THEN 'decrease'
    ELSE 'no_change'
  END AS change_flag
FROM rates;


In [0]:
%sql
-- Produtos que tiveram aumento (em algum mês)
SELECT
  product_code,
  month,
  abandon_rate,
  prev_abandon_rate,
  abandon_rate_diff
FROM vw_product_abandon_mom
WHERE change_flag = 'increase'
ORDER BY abandon_rate_diff DESC, abandoned_carts DESC;


## 11) 2.4 Relatório diário de abandono + Export TXT (Top 50 carrinhos)
Nesta etapa vamos:

1) Gerar um **relatório por data** com:
- **qtd_carrinhos_abandonados**
- **qtd_itens_abandonados** (soma das quantidades em `tb_cartentries`)
- **valor_nao_faturado** (soma de `tb_carts.p_totalprice` para carrinhos abandonados)

2) Exportar um **arquivo `.txt`** com os **50 carrinhos abandonados** de maior `p_totalprice` no layout:
`carts.PK|carts.createdTS|carts.p_totalprice|user.p_uid|payment`

> Observação: vamos reutilizar a view **`vw_base_carts`** (carts com `cart_status = abandoned/completed`) que já criamos.


### 11.1) Relatório por data (abandonos)
Definições:
- Carrinho abandonado = `vw_base_carts.cart_status = 'abandoned'`
- Itens abandonados = `SUM(tb_cartentries.p_quantity)` desses carrinhos
- Valor não faturado = `SUM(vw_base_carts.p_totalprice)` desses carrinhos


In [0]:
%sql
WITH abandoned AS (
  SELECT
    PK AS cart_id,
    CAST(createdTS AS DATE) AS dt,
    p_totalprice
  FROM vw_base_carts
  WHERE cart_status = 'abandoned'
    AND createdTS IS NOT NULL
),
items_per_cart AS (
  SELECT
    p_order AS cart_id,
    SUM(COALESCE(p_quantity, 0)) AS qtd_itens
  FROM delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_cartentries`
  GROUP BY p_order
)
SELECT
  a.dt AS data,
  COUNT(DISTINCT a.cart_id) AS qtd_carrinhos_abandonados,
  SUM(COALESCE(i.qtd_itens, 0)) AS qtd_itens_abandonados,
  SUM(COALESCE(a.p_totalprice, 0)) AS valor_nao_faturado
FROM abandoned a
LEFT JOIN items_per_cart i
  ON a.cart_id = i.cart_id
GROUP BY a.dt
ORDER BY a.dt;


### 11.2) Temp view com Top 50 carrinhos abandonados (linha pronta para TXT)
Layout exigido:
`carts.PK|carts.createdTS|carts.p_totalprice|user.p_uid|payment`

Aqui `payment` vai ser o **cart_status**:
- `abandoned` (sem paymentinfo no join)
- `completed` (com paymentinfo no join)

Como o TXT pede os 50 maiores `p_totalprice`, filtramos **abandoned** e ordenamos desc.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_top50_abandoned_txt_full AS
WITH base AS (
  SELECT
    c.PK,
    c.createdTS,
    c.p_totalprice,
    c.cart_status,
    c.p_user,
    c.p_paymentmode,
    c.p_paymentinfo,
    c.p_site,
    c.p_paymentaddress
  FROM vw_base_carts c
  WHERE c.cart_status = 'abandoned'
),
items AS (
  SELECT
    e.p_order AS cart_id,
    SUM(COALESCE(e.p_quantity, 0)) AS sum_quantity,
    COUNT(e.PK) AS count_items
  FROM delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_cartentries` e
  GROUP BY e.p_order
)
SELECT
  CONCAT(
    CAST(b.PK AS STRING), '|',
    COALESCE(CAST(b.createdTS AS STRING), ''), '|',
    COALESCE(CAST(b.p_totalprice AS STRING), ''), '|',
    COALESCE(u.p_uid, ''), '|',
    COALESCE(b.cart_status, ''), '|',
    COALESCE(pm.p_code, ''), '|',
    COALESCE(CAST(pi.p_installments AS STRING), ''), '|',
    COALESCE(site.p_name, ''), '|',
    COALESCE(addr.p_postalcode, ''), '|',
    COALESCE(CAST(it.sum_quantity AS STRING), '0'), '|',
    COALESCE(CAST(it.count_items AS STRING), '0')
  ) AS line
FROM base b
LEFT JOIN delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_users` u
  ON b.p_user = u.PK
LEFT JOIN delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_paymentmodes` pm
  ON b.p_paymentmode = pm.PK
LEFT JOIN delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_paymentinfos` pi
  ON b.p_paymentinfo = pi.PK
LEFT JOIN delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_cmssitelp` site
  ON b.p_site = site.ITEMPK
LEFT JOIN delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_addresses` addr
  ON b.p_paymentaddress = addr.PK
LEFT JOIN items it
  ON b.PK = it.cart_id
ORDER BY b.p_totalprice DESC
LIMIT 50;


In [0]:
%sql
-- Preview das linhas (para conferir antes de exportar)
SELECT line FROM vw_top50_abandoned_txt;


### 11.3) Exportar o TXT (um arquivo único)
Vamos coletar as 50 linhas e gravar em um `.txt` no DBFS.


In [0]:
out_path = "dbfs:/Volumes/workspace/cantustore/gold_prova_dados/exports/top50_abandoned.txt"

df_txt = spark.sql("SELECT line FROM vw_top50_abandoned_txt_full")
df_txt.coalesce(1).write.mode("overwrite").text(out_path)

print("OK ->", out_path)
